In [55]:
import sys
import re
import numbers

import pandas as pd
import numpy as np

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from read_and_write_docs import read_rds, read_excel_sheets

In [56]:
base_location = "/Volumes/BCross/av_datasets_experiments/concat_ngram_masking_logprobs_gc/_results_v2"
# base_location = "/Users/user/Documents/uni_work_offline/av_datasets_experiments/concat_ngram_masking_logprobs_gc/_results_v2"

file_loc = f"{base_location}/performance_summary_by_token_calibrated_raw.rds"

table_save_loc = f"{base_location}/tables"

baselines = read_excel_sheets('/Users/user/Documents/PhD Stuff/Base Idiolect Results Table V2.xlsx')
baselines = baselines['All Results']
baselines['row_number'] = (
    baselines.groupby('Corpus').cumcount() + 1
)
baselines['method_type'] = 'baseline'

df = read_rds(file_loc)

In [57]:
baselines

,Corpus,Method,Acc.,AUC,F1,Prec.,Rec.,TP,FN,FP,TN,row_number,method_type
0,C_Yelp,LambdaG,0.765,0.839,0.769,0.783,0.755,188,52,61,179,1,baseline
1,C_Yelp,IM,0.650,0.735,0.613,0.554,0.686,133,107,61,179,2,baseline
2,C_Yelp,COAV,0.638,0.728,0.640,0.646,0.635,155,85,89,151,3,baseline
3,C_Yelp,TAVeer,0.671,0.749,0.686,0.721,0.655,173,67,91,149,4,baseline
4,C_Yelp,FeVecDiff (LR),0.508,0.514,0.561,0.507,0.629,151,89,147,93,5,baseline
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,C_Trip,FeVecDiff (MLP),0.477,0.475,0.516,0.480,0.558,134,106,145,95,6,baseline
116,C_Trip,ADHOMINEM,0.494,0.497,0.475,0.493,0.458,110,130,113,127,7,baseline
117,C_Trip,ADHOMINEM (†),0.502,0.525,0.505,0.502,0.508,122,118,121,119,8,baseline
118,C_Trip,LUAR,0.667,0.755,0.692,0.643,0.750,180,60,100,140,9,baseline


In [58]:
baselines['Corpus'].unique()

array(['C_Yelp', 'C_IMDB', 'C_Blog', 'C_Amazon', 'C_AllNews', 'C_Enron',
       'C_Wiki', 'C_Stack', 'C_ACL', 'C_PJ', 'C_Apric', 'C_Trip'],
      dtype=object)

In [59]:
# Just keep the main scoring column as most consistently accurate
scored_df = df[
    df['scoring_col'].isin(['unknown_sum_log_probs', 'unknown_sum_log_probs_sum'])
].copy()

# We only want to keep the corpora in the LambdaG paper
corpus_dict = {
    "Enron Cleaned": "C_Enron",
    "Wiki": "C_Wiki",
    "StackExchange": "C_Stack",
    "ACL": "C_ACL",
    "Perverted Justice": "C_PJ",
    "The Apricity": "C_Apric",
    "TripAdvisor": "C_Trip",
    "Yelp": "C_Yelp",
    "IMDB": "C_IMDB",
    "Koppel's Blogs": "C_Blog",
    "Amazon": "C_Amazon",
    "All-the-news": "C_AllNews"
}

filtered_df = scored_df[
    scored_df['corpus'].isin(corpus_dict)
    & (scored_df['min_token_size'].isin([3]))
    & (scored_df['max_context_tokens'].isin([10]))
    & (scored_df['scoring_col'] == "unknown_sum_log_probs")
].copy()

# Replace each corpus name with its corresponding abbreviated value
filtered_df['corpus'] = filtered_df['corpus'].map(corpus_dict)

In [60]:
# Fix Scoring Column
filtered_df["scoring_alias"] = np.select(
    [
        filtered_df["scoring_col"] == "unknown_sum_log_probs",
        filtered_df["scoring_col"] == "unknown_sum_log_probs_sum",
        filtered_df["scoring_col"] == "llr_unknown_vs_nocontext",
        filtered_df["scoring_col"] == "llr_unknown_vs_nocontext_sum",
    ],
    [
        "UnkAvg",
        "UnkSum",
        "UnkVsNcAvg",
        "UnkVsNcSum",
    ],
    default="unknown"
)

In [61]:
def prep_results(df, prefix, keep_descriptors=False):
    out = df.copy()

    # keep only min_token_size 3, or 4
    out = out[out["min_token_size"].isin([3, 4])].copy()

    # convert to int once
    out["max_context_tokens"] = out["max_context_tokens"].astype(int)
    out["min_token_size"] = out["min_token_size"].astype(int)

    # work out zero-padding widths from the largest value in each column
    max_context_width = out["max_context_tokens"].astype(str).str.len().max()
    min_token_width = out["min_token_size"].astype(str).str.len().max()

    out["Method"] = (
        prefix
        + "_"
        + out["scoring_alias"].astype(str)
        + "_"
        + out["max_context_tokens"].astype(str).str.zfill(max_context_width)
        + "_"
        + out["min_token_size"].astype(str).str.zfill(min_token_width)
    )

    # round metrics
    round_cols = ["Balanced Accuracy", "AUC", "F1", "Precision", "Recall"]
    out[round_cols] = out[round_cols].round(3)

    if keep_descriptors:
        out = out[
            ["corpus", "Method", "min_token_size", "max_context_tokens",
             "Balanced Accuracy", "AUC", "F1", "Precision", "Recall",
             "TP", "FN", "FP", "TN"]
        ].copy()

        out = (
            out
            .rename(columns={
                "corpus": "Corpus",
                "min_token_size": "Min Token Size",
                "max_context_tokens": "Max Context Tokens"
            })
            .sort_values(["Corpus", "Min Token Size", "Max Context Tokens"])
            .reset_index(drop=True)
        )
    else:
        out = out[
            ["corpus", "Method", "Balanced Accuracy", "AUC", "F1",
             "Precision", "Recall", "TP", "FN", "FP", "TN"]
        ].copy()

        out = (
            out
            .rename(
                columns={
                    "corpus": "Corpus",
                    "Balanced Accuracy": "Acc.",
                    "Precision": "Prec.",
                    "Recall": "Rec."
                }
            )
            .sort_values(["Corpus", "Method"])
            .reset_index(drop=True)
        )

    return out

In [62]:
results = prep_results(filtered_df, prefix="LPTracing")
results['row_number'] = range(1, len(results) + 1)
results['method_type'] = 'my_experiment'

results["Method"] = results["Method"].str.replace(
    r"_0+_(\d+)$",
    r"_ContextFree_\1",
    regex=True
)

In [63]:
results

,Corpus,Method,Acc.,AUC,F1,Prec.,Rec.,TP,FN,FP,TN,row_number,method_type
0,C_ACL,LPTracing_UnkAvg_10_3,0.698,0.733,0.656,0.767,0.572,79,59,24,112,1,my_experiment
1,C_AllNews,LPTracing_UnkAvg_10_3,0.761,0.831,0.737,0.853,0.648,786,427,135,937,2,my_experiment
2,C_Amazon,LPTracing_UnkAvg_10_3,0.763,0.841,0.754,0.785,0.725,870,330,238,961,3,my_experiment
3,C_Apric,LPTracing_UnkAvg_10_3,0.796,0.876,0.794,0.801,0.787,133,36,33,136,4,my_experiment
4,C_Blog,LPTracing_UnkAvg_10_3,0.763,0.838,0.755,0.783,0.729,656,244,182,713,5,my_experiment
5,C_Enron,LPTracing_UnkAvg_10_3,0.801,0.882,0.795,0.833,0.761,35,11,7,37,6,my_experiment
6,C_IMDB,LPTracing_UnkAvg_10_3,0.673,0.722,0.642,0.761,0.554,392,315,123,467,7,my_experiment
7,C_PJ,LPTracing_UnkAvg_10_3,0.849,0.899,0.836,0.916,0.769,120,36,11,143,8,my_experiment
8,C_Stack,LPTracing_UnkAvg_10_3,0.588,0.602,0.546,0.609,0.496,56,57,36,77,9,my_experiment
9,C_Trip,LPTracing_UnkAvg_10_3,0.634,0.675,0.577,0.721,0.481,101,109,39,145,10,my_experiment


In [64]:
combined_results = (
    pd.concat([baselines, results], ignore_index=True)
    .sort_values(
        by=["Corpus", "method_type", "row_number"]
    )
    .reset_index(drop=True)
)

# Keep this before dropping method_type
combined_results["__highlight__"] = (
    combined_results["method_type"].eq("my_experiment")
)

combined_results = combined_results.drop(
    columns=["method_type", "row_number"]
)


def latex_escape(x):
    """
    Escape normal text for LaTeX.
    """
    if pd.isna(x):
        return ""

    x = str(x)

    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
        "†": r"\textdagger{}",
    }

    for old, new in replacements.items():
        x = x.replace(old, new)

    return x


def corpus_to_math_label(corpus):
    """
    Converts corpus names to labels like:
    Yelp -> $C_{\\mathrm{Yelp}}$
    C_Yelp -> $C_{\\mathrm{Yelp}}$
    """

    corpus = str(corpus)

    corpus_labels = {
        "Yelp": r"$C_{\mathrm{Yelp}}$",
        "IMDB": r"$C_{\mathrm{IMDB}}$",
        "Blog": r"$C_{\mathrm{Blog}}$",
        "Blogger": r"$C_{\mathrm{Blog}}$",
        "Amazon": r"$C_{\mathrm{Amazon}}$",
        "AllNews": r"$C_{\mathrm{AllNews}}$",
        "Enron": r"$C_{\mathrm{Enron}}$",
        "Wiki": r"$C_{\mathrm{Wiki}}$",
        "Stack": r"$C_{\mathrm{Stack}}$",
        "Stack.": r"$C_{\mathrm{Stack}}$",
        "ACL": r"$C_{\mathrm{ACL}}$",
        "PJ": r"$C_{\mathrm{PJ}}$",
        "Apric": r"$C_{\mathrm{Apric}}$",
        "Trip": r"$C_{\mathrm{Trip}}$",
    }

    if corpus in corpus_labels:
        return corpus_labels[corpus]

    # fallback for names like C_Yelp
    corpus = re.sub(r"^C[_\s-]*", "", corpus)

    # remove awkward punctuation for the subscript
    corpus = re.sub(r"[^A-Za-z0-9]", "", corpus)

    return rf"$C_{{\mathrm{{{corpus}}}}}$"


def format_latex_cell(value, column):
    """
    Format numbers and text for LaTeX.
    """
    if pd.isna(value):
        return ""

    count_cols = {"TP", "FN", "FP", "TN"}

    if column in count_cols:
        return str(int(round(float(value))))

    if isinstance(value, numbers.Number):
        return f"{value:.3f}"

    return latex_escape(value)


columns = [
    col for col in combined_results.columns
    if col != "__highlight__"
]

n_cols = len(columns)

column_format = (
    "@{}"
    + "c"      # Corpus column
    + "l"      # Method column
    + "c" * (n_cols - 2)
    + "@{}"
)

header = " & ".join(latex_escape(col) for col in columns) + r" \\"

latex_lines = [
    r"\begin{longtable}{" + column_format + r"}",
    r"\caption{Results}",
    r"\label{tab:results}\\",
    r"\toprule",
    header,
    r"\midrule",
    r"\endfirsthead",
    r"\caption[]{Results continued}\\",
    r"\toprule",
    header,
    r"\midrule",
    r"\endhead",
    r"\midrule",
    rf"\multicolumn{{{n_cols}}}{{r}}{{Continued on next page}}\\",
    r"\endfoot",
    r"\bottomrule",
    r"\endlastfoot",
]

for corpus_i, (corpus, corpus_df) in enumerate(
    combined_results.groupby("Corpus", sort=False)
):
    if corpus_i > 0:
        latex_lines.append(r"\midrule")

    n_corpus_rows = len(corpus_df)

    corpus_cell = (
        rf"\multirow[c]{{{n_corpus_rows}}}{{*}}"
        rf"{{\rotatebox[origin=c]{{90}}{{{corpus_to_math_label(corpus)}}}}}"
    )

    for row_i, (_, row) in enumerate(corpus_df.iterrows()):

        if row["__highlight__"]:
            latex_lines.append(r"\rowcolor{gray!8}")

        first_cell = corpus_cell if row_i == 0 else ""

        row_cells = [first_cell]

        for col in columns[1:]:
            row_cells.append(format_latex_cell(row[col], col))

        # Important: stop longtable splitting the multirow corpus block
        row_end = r" \\*" if row_i < n_corpus_rows - 1 else r" \\"

        latex_lines.append(" & ".join(row_cells) + row_end)

latex_lines.append(r"\end{longtable}")

latex_table = "\n".join(latex_lines)

latex_table = (
    "\\begingroup\n"
    "\\small\n"
    "\\setlength{\\tabcolsep}{4pt}\n"
    "\\setlength{\\LTleft}{-0.5cm}\n"
    "\\setlength{\\LTright}{\\fill}\n"
    f"{latex_table}\n"
    "\\endgroup\n"
)

In [65]:
with open(f"{table_save_loc}/results_table.tex", "w", encoding="utf-8") as file:
    file.write(latex_table)